# Pinned OCSF export
Export a verified evidence bundle as core OCSF 1.3.0. Schema validation runs offline, uses zero model tokens, and keeps the original bundle unchanged. Required fields are never filled with model guesses.

In [ ]:
from pathlib import Path
import tempfile
import json
root = Path.cwd()
if not (root / 'examples').exists():
    root = root.parent
assert (root / 'examples/parser_samples.json').exists(), 'Run from the repository or notebooks directory'
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.core.manifest import verify_bundle
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
bundle = workdir / 'bundle'
manifest = run_pipeline([
    Input('cloudtrail', root / 'examples/raw/aws/cloudtrail_real_sample.json'),
    Input('entra_signin', root / 'examples/raw/entra/entra_signin_real_sample.jsonl'),
    Input('crowdstrike_detection', root / 'examples/raw/edr/crowdstrike_detection_real_sample.json'),
], bundle, 'notebook-demo')


In [ ]:
from timeline_demo.ocsf import export_bundle, verify_export, schema_lock
from timeline_demo.parsers.common import file_hash
source_pin = file_hash(bundle/'audit_manifest.json')
ocsf_dir = workdir/'ocsf'
report = export_bundle(bundle, ocsf_dir, manifest_sha256=source_pin)
assert report['counts'] == {'source_events':5, 'exported_events':5, 'rejected_events':0}
assert report['model_tokens'] == 0
report['counts']

In [ ]:
lock = schema_lock()
[(int(uid), entry['name']) for uid, entry in lock['classes'].items()]

In [ ]:
export_pin = file_hash(ocsf_dir/'export_manifest.json')
assert verify_export(ocsf_dir, bundle=bundle, manifest_sha256=export_pin) == report
ocsf_rows = [json.loads(line) for line in (ocsf_dir/'ocsf.jsonl').read_text().splitlines()]
ocsf_rows[0]

## Explicitly account for sparse source records
The timeline can retain a partial audit record, but the API Activity export requires an actor and source endpoint. Strict mode rejects the export; quarantine mode writes a rejection receipt. Neither mode changes the evidence bundle.

In [ ]:
sparse = workdir/'sparse.json'
sparse.write_text(json.dumps({'eventTime':'2026-09-01T10:00:00Z','eventName':'GetObject','eventID':'sparse-1'}))
sparse_bundle = workdir/'sparse_bundle'
run_pipeline([Input('cloudtrail', sparse)], sparse_bundle, 'sparse-demo')
partial = export_bundle(sparse_bundle, workdir/'partial', quarantine=True)
assert partial['counts']['rejected_events'] == 1
json.loads((workdir/'partial/rejections.jsonl').read_text())

## Databricks publication
On a configured Databricks cluster, generate the export into a separate Unity Catalog Volume directory, then call `publish_ocsf_export(spark, export_dir, source_bundle, catalog, schema)`. Both paths must be accessible to Spark workers. The function verifies the source binding, inserts events and rejection receipts, then writes a final marker. Query `published_ocsf` for completed exports and inspect `published_ocsf_exports` for rejection counts. See `docs/OCSF_EXPORT.md` for a runnable Volume example and Tines completion implications. A successful export with quarantine enabled may be partial.

In [ ]:
assert file_hash(bundle/'audit_manifest.json') == source_pin
work.cleanup()